# Task 1 — Inconsistencies and duplicates

**Daniel Ehab** · 58-3825 · MET-02 · CSEN1095 Data Engineering, W26

Student clubs collected sign-ups through an online form for one week, and `club_signups.csv` holds one row per form submission. This notebook cleans it into a table with **one row per student per club** and every value spelled one way, then saves it as `club_signups_clean.csv`.

The notebook follows the Lab 3 rules:

1. **The raw file is never edited.** It is read once into `raw`, and every fix happens on a copy.
2. **Every fix is code.** Each fix is followed by a ✓ check cell that fails loudly if the fix did not work.
3. **Count before and after.** Every step that changes rows says how many.

## Setup

The course's pinned environment from Lab 1: Python 3.12 and pandas 2.3.3. The versions are printed so that anyone who re-runs the notebook can compare.

In [1]:
import sys

import pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)

Python 3.12.14
pandas 2.3.3


## Part A — Inconsistencies

### A.1 Look first

Before fixing anything, we answer the Lab 3 first-look questions for this file: what one row is, whether the key is unique, what type each column has, what the categories look like, and where the nulls are.

The file is read once into `raw`. Nothing assigns to `raw` after this cell; every fix later works on a copy.

In [2]:
raw = pd.read_csv("club_signups.csv")

print(raw.shape)
raw.head()

(39, 9)


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Soccer,2026-09-15 13:39,N
1,S010,61-3846,AHMED ZAKI,ahmed.zaki@student.guc.edu.eg,Management,alexandria,Chess,15/09/2026 03:12,false
2,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32,Y
3,S035,61-4844,MOHAMED ADEL,mohamed.adel@student.guc.edu.eg,Engineering and Materials Science,Cairo,Debating,2026-09-17 11:15,yes
4,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess Club,2026-09-13 22:36,yes


**Types.** Which type did pandas give each column?

In [3]:
raw.dtypes

signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

**Distinct values and nulls.** How many different values does each column hold, and is anything missing?

In [4]:
pd.DataFrame({"distinct": raw.nunique(), "missing": raw.isna().sum()})

,distinct,missing
signup_id,36,0
student_id,15,0
full_name,24,0
email,18,0
faculty,13,0
city,9,0
club,13,0
signed_up_at,36,0
fee_paid,9,0


**The key.** The form gave every submission a `signup_id`. Is it unique?

In [5]:
print("rows:", len(raw), "| distinct signup_id:", raw["signup_id"].nunique(), "| unique?", raw["signup_id"].is_unique)

rows: 39 | distinct signup_id: 36 | unique? False


**The categories.** `value_counts()` of every categorical column, including any nulls (`dropna=False`).

In [6]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(raw[col].value_counts(dropna=False), "\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO         

**What we see**, and where each problem gets fixed:

- **One row is one form submission.** 39 rows and 9 columns, as the brief says, and no value is missing anywhere.
- **Every column is text** (`object`). `fee_paid` should be True/False (→ A.4) and `signed_up_at` should be a datetime (→ A.5). The first five rows already show two date shapes: `2026-09-15 13:39` and `15/09/2026 03:12`.
- **The key repeats.** 39 rows but only 36 distinct `signup_id`s, so 3 rows reuse an id that already exists (→ B.1).
- **Too many spellings.** `faculty` has 13 distinct values where 5 are allowed, `club` 13 for 5, and `city` 9 for 3 (→ A.2). The variants come in three kinds: case (`iet`, `CAIRO`), longer or shorter names (`Information Engineering and Technology`, `Pharma`, `Chess Club`, `Alex`), and synonyms (`Soccer`, `Debating`).
- **A value that prints twice.** `Music` appears twice in the `club` counts (5 and 1). Two values that print the same must differ in something invisible (→ checked in A.2).
- **One person, several spellings.** 15 distinct `student_id`s but 24 distinct names and 18 distinct emails, e.g. `AHMED ZAKI` and `MOHAMED ADEL` in capitals (→ A.3).
- **Nine ways to write yes or no** in `fee_paid`: `1`, `yes`, `Y`, `TRUE`, `Yes`, `false`, `N`, `0`, `no` (→ A.4).

### A.2 Faculty, club and city

A.1 showed far more distinct values than the brief allows: 13 faculties for 5, 13 clubs for 5, 9 cities for 3. It also showed `Music` twice in the club counts. We follow the Lab 3 recipe for one spelling per value: look, normalise, map, assert.

**Look closer.** `value_counts()` prints values without quotes, so two values that differ only by a space look identical. Printed as a Python list, each value shows its quotes.

In [7]:
for col in ["faculty", "club", "city"]:
    print(col, sorted(raw[col].unique()), "\n")

faculty ['EMS', 'Engineering and Materials Science', 'IET', 'Information Engineering and Technology', 'MET', 'Management', 'Media Engineering and Technology', 'Pharma', 'Pharmacy', 'ems', 'iet', 'met', 'pharmacy'] 

club ['Chess', 'Chess Club', 'Debate', 'Debating', 'Football', 'Music', 'Music ', 'Robotics', 'Soccer', 'chess', 'debate club', 'football', 'music'] 

city ['Alex', 'Alexandria', 'CAIRO', 'Cairo', 'El Giza', 'Giza', 'alexandria', 'cairo', 'giza'] 



`'Music '` ends with a space: that is the second `Music` from A.1. The other variants differ in case (`iet`, `CAIRO`), in length (`Pharma`, `Chess Club`, `Alex`), or are synonyms (`Soccer`, `Debating`).

**Normalise mechanically.** Stripping spaces and lower-casing fixes the space and every case variant at once. From here on every fix happens on `signups`, a copy of `raw`.

In [8]:
signups = raw.copy()

for col in ["faculty", "club", "city"]:
    before = signups[col].nunique()
    signups[col] = signups[col].str.strip().str.lower()
    print(f"{col}: {before} distinct -> {signups[col].nunique()} after strip + lower")
    print(sorted(signups[col].unique()), "\n")

faculty: 13 distinct -> 9 after strip + lower
['ems', 'engineering and materials science', 'iet', 'information engineering and technology', 'management', 'media engineering and technology', 'met', 'pharma', 'pharmacy'] 

club: 13 distinct -> 9 after strip + lower
['chess', 'chess club', 'debate', 'debate club', 'debating', 'football', 'music', 'robotics', 'soccer'] 

city: 9 distinct -> 5 after strip + lower
['alex', 'alexandria', 'cairo', 'el giza', 'giza'] 



Normalising took `faculty` from 13 to 9 distinct values, `club` from 13 to 9 and `city` from 9 to 5. What is left cannot be fixed mechanically: full names (`information engineering and technology`), short forms (`pharma`, `alex`), a `club` suffix, and synonyms (`soccer`, `debating`).

**Map what is left** with one explicit dictionary per column, with one entry for every spelling printed above. The brief gives two of them: Media Engineering and Technology is MET, and a Soccer club is the Football club. Lower-casing also turned `MET` into `met`, so the canonical spellings need an entry too.

We use `.map()` rather than `.replace()`, so the dictionary is the complete list of accepted spellings: a spelling missing from it becomes empty (`NaN`) instead of staying in the column looking valid. Either way, the check below fails if anything outside the allowed values is left.

In [9]:
FACULTIES = {"MET", "IET", "EMS", "Pharmacy", "Management"}
CLUBS = {"Robotics", "Debate", "Music", "Football", "Chess"}
CITIES = {"Cairo", "Giza", "Alexandria"}

FACULTY_MAP = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
CLUB_MAP = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
CITY_MAP = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

signups["faculty"] = signups["faculty"].map(FACULTY_MAP)
signups["club"] = signups["club"].map(CLUB_MAP)
signups["city"] = signups["city"].map(CITY_MAP)

In [10]:
# ✓ check: only the canonical values remain, and no row was lost
assert len(signups) == len(raw), "a row went missing"
for col, allowed in [("faculty", FACULTIES), ("club", CLUBS), ("city", CITIES)]:
    assert signups[col].isin(allowed).all(), f"{col} still holds {set(signups[col]) - allowed}"
    print(col, signups[col].value_counts().to_dict())
print("✓ faculty, club and city hold only the canonical values")

faculty {'IET': 12, 'Pharmacy': 9, 'EMS': 9, 'MET': 5, 'Management': 4}
club {'Football': 9, 'Chess': 9, 'Music': 8, 'Robotics': 8, 'Debate': 5}
city {'Alexandria': 16, 'Cairo': 16, 'Giza': 7}
✓ faculty, club and city hold only the canonical values


**One more proof.** A student has one faculty and one city, however they typed them. If the dictionaries are right, every `student_id` now shows exactly one of each.

In [11]:
# ✓ check: each student now has a single faculty and a single city
per_student = signups.groupby("student_id")[["faculty", "city"]].nunique()
assert (per_student == 1).all().all(), per_student[(per_student > 1).any(axis=1)]
print(f"✓ all {len(per_student)} students have one faculty and one city")

✓ all 15 students have one faculty and one city


These counts are still per sign-up row, 39 of them, duplicates included. Part B gets the table to one row per student per club.

### A.3 Names and emails

A.1 found 24 distinct names and 18 distinct emails for 15 students, and A.2 showed that this file hides spaces at the edges of values. The brief asks for one spelling per person.

**Look.** How many spellings does each student have? For every student with more than one, the spellings are printed as Python lists, so spaces show.

In [12]:
spellings = signups.groupby("student_id")[["full_name", "email"]].nunique()
messy = spellings[(spellings > 1).any(axis=1)].index
print(f"{len(messy)} of {len(spellings)} students have more than one spelling\n")

for sid in messy:
    rows = signups[signups["student_id"] == sid]
    print(sid, sorted(rows["full_name"].unique()))
    print(" " * len(sid), sorted(rows["email"].unique()), "\n")

7 of 15 students have more than one spelling

55-2992 ['Mohamed Adel', 'mohamed adel']
        ['mohamed.adel@student.guc.edu.eg'] 

55-9245 ['ALI IBRAHIM', 'Ali Ibrahim', 'ali ibrahim']
        ['ali.ibrahim@student.guc.edu.eg'] 

58-8701 ['  Farida  Ibrahim ', 'Farida Ibrahim']
        ['farida.ibrahim@student.guc.edu.eg'] 

61-2495 ['AHMED ADEL', 'ahmed adel']
        ['AHMED.ADEL@STUDENT.GUC.EDU.EG', 'ahmed.adel@student.guc.edu.eg'] 

61-4733 ['  Omar  Adel ', 'OMAR ADEL']
        ['omar.adel@student.guc.edu.eg'] 

61-4844 ['  Mohamed  Adel ', 'MOHAMED ADEL', 'Mohamed Adel']
        [' mohamed.adel@student.guc.edu.eg ', 'MOHAMED.ADEL@STUDENT.GUC.EDU.EG', 'mohamed.adel@student.guc.edu.eg'] 

61-7344 ['  Laila  Mostafa ', 'Laila Mostafa', 'laila mostafa']
        ['LAILA.MOSTAFA@STUDENT.GUC.EDU.EG', 'laila.mostafa@student.guc.edu.eg'] 



7 of the 15 students have more than one spelling. Names differ in case (`ALI IBRAHIM`, `ali ibrahim`) and in spaces, and `'  Farida  Ibrahim '` has spaces at both ends **and** two in the middle. Emails differ in case, and one has a space at each end.

**Fix.** `strip()` only trims the ends, so it cannot remove the double space in the middle. Instead, `split()` breaks a name on any run of spaces and drops the ones at the edges, and `" ".join` puts it back together with single spaces. `title()` then gives everyone the same capitalisation. Emails only need trimming and lower-case.

In [13]:
names_before, emails_before = signups["full_name"].nunique(), signups["email"].nunique()

signups["full_name"] = signups["full_name"].str.split().str.join(" ").str.title()
signups["email"] = signups["email"].str.strip().str.lower()

print(f"distinct names:  {names_before} -> {signups['full_name'].nunique()}")
print(f"distinct emails: {emails_before} -> {signups['email'].nunique()}")

distinct names:  24 -> 14
distinct emails: 18 -> 14


In [14]:
# ✓ check: one spelling per person
assert (signups.groupby("student_id")[["full_name", "email"]].nunique() == 1).all().all(), "a student still has two spellings"
assert not signups["full_name"].str.contains(r"^\s|\s$|\s\s").any(), "a name still has extra spaces"
assert (signups["full_name"] == signups["full_name"].str.title()).all(), "a name is capitalised differently"
assert (signups["email"] == signups["email"].str.strip().str.lower()).all(), "an email is not trimmed and lower-case"
print("✓ every student has one name and one email; no extra spaces; same capitalisation; emails trimmed and lower-case")

✓ every student has one name and one email; no extra spaces; same capitalisation; emails trimmed and lower-case


In [15]:
# the messiest student from the look above, before and after
pd.DataFrame({"before": raw.loc[raw["student_id"] == "61-4844", "full_name"].map(repr),
              "after": signups.loc[signups["student_id"] == "61-4844", "full_name"].map(repr)})

,before,after
2,' Mohamed Adel ','Mohamed Adel'
3,'MOHAMED ADEL','Mohamed Adel'
22,'Mohamed Adel','Mohamed Adel'
23,'Mohamed Adel','Mohamed Adel'


Every student now has one spelling, yet there are 14 distinct names and 14 distinct emails for 15 students.

**Flip the question.** Does each name, and each email, belong to only one `student_id`?

In [16]:
students_per_name = signups.groupby("full_name")["student_id"].nunique()
students_per_email = signups.groupby("email")["student_id"].nunique()
print("names used by more than one student_id: ", students_per_name[students_per_name > 1].to_dict())
print("emails used by more than one student_id:", students_per_email[students_per_email > 1].to_dict())

shared = students_per_name[students_per_name > 1].index
signups.loc[signups["full_name"].isin(shared), ["student_id", "full_name", "email", "faculty", "city"]].drop_duplicates()

names used by more than one student_id:  {'Mohamed Adel': 2}
emails used by more than one student_id: {'mohamed.adel@student.guc.edu.eg': 2}


,student_id,full_name,email,faculty,city
2,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo
14,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria


**Two people, one name.** `Mohamed Adel` belongs to two `student_id`s: 61-4844 studies EMS and lives in Cairo, 55-2992 studies IET and lives in Alexandria. They even share the email `mohamed.adel@student.guc.edu.eg`. Different ids, faculties and cities: these are two students.

So neither the name nor the email can tell people apart; only `student_id` can. Part B therefore identifies a person by `student_id` alone, and Part C shows with numbers what a name-based key would have done.

### A.4 fee_paid

A.1 showed that `fee_paid` is text (`object`) with nine spellings of yes and no. The brief wants a boolean column: True or False.

**Look.** The spellings after the same mechanical step as in A.2 (strip and lower-case), printed with quotes, and how many rows use each.

In [17]:
fee_spellings = signups["fee_paid"].str.strip().str.lower().value_counts()
print(f"{signups['fee_paid'].nunique()} raw spellings -> {len(fee_spellings)} after strip + lower")
print({repr(k): v for k, v in fee_spellings.items()})

9 raw spellings -> 8 after strip + lower
{"'yes'": 11, "'1'": 7, "'false'": 6, "'y'": 5, "'n'": 4, "'true'": 4, "'0'": 1, "'no'": 1}


Lower-casing merged `Yes` and `yes`, leaving 8 spellings: four that mean yes (`yes`, `1`, `y`, `true`) and four that mean no (`false`, `n`, `0`, `no`).

**The tempting shortcut.** `astype(bool)` looks like the obvious way to get a boolean column. Tried on the column without saving the result:

In [18]:
signups["fee_paid"].astype(bool).value_counts()

fee_paid
True    39
Name: count, dtype: int64

Every row comes out True, including `no`, `false` and `0`: for text, `astype(bool)` only asks whether the string is non-empty. So the meaning has to come from an explicit dictionary.

**Fix.** Map each spelling to True or False. As in A.2, `.map()` turns any spelling missing from the dictionary into `NaN`, which the check below catches. We do not call `astype(bool)` afterwards, because it would quietly turn such a `NaN` into True.

In [19]:
FEE_MAP = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}

signups["fee_paid"] = signups["fee_paid"].str.strip().str.lower().map(FEE_MAP)

In [20]:
# ✓ check: a real boolean column, every spelling known, totals as in the look above
assert signups["fee_paid"].notna().all(), "a fee spelling is missing from FEE_MAP"
assert signups["fee_paid"].dtype == bool, f"fee_paid is {signups['fee_paid'].dtype}"

paid = signups["fee_paid"].value_counts()
yes_spellings = [spelling for spelling, value in FEE_MAP.items() if value]
assert paid[True] == fee_spellings[yes_spellings].sum() and paid.sum() == len(signups)
print(f"✓ fee_paid is {signups['fee_paid'].dtype}: {paid[True]} True, {paid[False]} False")

✓ fee_paid is bool: 27 True, 12 False


`fee_paid` is now a boolean column: 27 True and 12 False, the same split as the spellings in the look. Like the counts in A.2, these are still per sign-up row; Part B will show what changes when resubmissions are removed.

### A.5 signed_up_at

A.1 showed that `signed_up_at` is text, and its first five rows already had two shapes. The brief says two formats are mixed in this column.

**First attempt.** Let pandas parse the column on its own. The error is caught and printed, so the notebook still runs top to bottom.

In [21]:
try:
    pd.to_datetime(signups["signed_up_at"])
except ValueError as error:
    print("ValueError:", error)

ValueError: time data "15/09/2026 03:12" doesn't match format "%Y-%m-%d %H:%M", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


pandas guessed a single format, `%Y-%m-%d %H:%M`, from the first row and applied it to every row. It broke at position 1, on `15/09/2026 03:12`. It suggests `format='mixed'`, which would guess row by row, but the brief asks us to find the formats and decide day and month ourselves.

**Find the formats.** Label every row with the pattern it matches, using `str.fullmatch`, so that nothing else can hide in the column.

In [22]:
when = signups["signed_up_at"]
is_iso = when.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}")    # 2026-09-15 13:39
is_slash = when.str.fullmatch(r"\d{2}/\d{2}/\d{4} \d{2}:\d{2}")  # 15/09/2026 03:12

print("NNNN-NN-NN NN:NN :", is_iso.sum(), "rows, e.g.", when[is_iso].iloc[0])
print("NN/NN/NNNN NN:NN :", is_slash.sum(), "rows, e.g.", when[is_slash].iloc[0])
print("both or neither  :", (is_iso == is_slash).sum(), "rows")

NNNN-NN-NN NN:NN : 25 rows, e.g. 2026-09-15 13:39
NN/NN/NNNN NN:NN : 14 rows, e.g. 15/09/2026 03:12
both or neither  : 0 rows


**Day or month?** The first pattern starts with a four-digit year, so it reads year-month-day. In the slash pattern either of the first two numbers could be the day. A month cannot go above 12, so we look at the range of each part.

In [23]:
first = when[is_slash].str[:2].astype(int)
second = when[is_slash].str[3:5].astype(int)
print(f"slash, first part:  {first.min()} to {first.max()}, above 12 in {(first > 12).sum()} of {len(first)} rows")
print(f"slash, second part: {second.min()} to {second.max()}, above 12 in {(second > 12).sum()} of {len(second)} rows")
print("year-first rows, months:", sorted(when[is_iso].str[5:7].unique()), "| days:", when[is_iso].str[8:10].min(), "to", when[is_iso].str[8:10].max())

slash, first part:  13 to 18, above 12 in 14 of 14 rows
slash, second part: 9 to 9, above 12 in 0 of 14 rows
year-first rows, months: ['09'] | days: 13 to 17


Exactly two formats: 25 rows are year-first and 14 use slashes, and no row matches both or neither. In the slash rows the first number runs from 13 to 18, above 12 in every row, so it cannot be the month: **it is the day**. The second number is always 9, September. That agrees with the year-first rows, which are all in September (days 13 to 17): read day-first, the slash rows fall in the same week, as the brief's "one week" says. Read month-first, they would be impossible dates.

**Parse.** Each group gets its own explicit format, and both go into one datetime column, as in the Lab 3 notebook. Writing the format down records the decision instead of leaving it to a guess.

In [24]:
parsed = pd.Series(pd.NaT, index=signups.index, dtype="datetime64[ns]")
parsed[is_iso] = pd.to_datetime(when[is_iso], format="%Y-%m-%d %H:%M")
parsed[is_slash] = pd.to_datetime(when[is_slash], format="%d/%m/%Y %H:%M")

signups["signed_up_at"] = parsed

In [25]:
# ✓ check: one datetime column, every value parsed, day and month the right way round
assert (is_iso != is_slash).all(), "a row matches no pattern, or both"
assert signups["signed_up_at"].notna().all(), "a time did not parse"
assert pd.api.types.is_datetime64_dtype(signups["signed_up_at"]), signups["signed_up_at"].dtype

s010 = signups.loc[signups["signup_id"] == "S010", "signed_up_at"].item()  # arrived as "15/09/2026 03:12"
assert s010 == pd.Timestamp("2026-09-15 03:12"), s010

print(f"✓ signed_up_at is {signups['signed_up_at'].dtype}: from {signups['signed_up_at'].min()} to {signups['signed_up_at'].max()}")

✓ signed_up_at is datetime64[ns]: from 2026-09-13 14:36:00 to 2026-09-18 23:36:00

In [26]:
# a few slash rows, as they arrived and as parsed
pd.DataFrame({"arrived as": raw.loc[is_slash, "signed_up_at"], "parsed": signups.loc[is_slash, "signed_up_at"]}).head(4)

,arrived as,parsed
1,15/09/2026 03:12,2026-09-15 03:12:00
5,15/09/2026 03:42,2026-09-15 03:42:00
11,14/09/2026 01:22,2026-09-14 01:22:00
12,16/09/2026 23:55,2026-09-16 23:55:00


Every sign-up now has a real time, from 13 to 18 September 2026. `S010`, which arrived as `15/09/2026 03:12`, reads 15 September: day first.

### Part A, before and after

The type and the number of distinct values of every column, in `raw` and in `signups`:

In [27]:
pd.DataFrame({
    "dtype before": raw.dtypes, "dtype after": signups.dtypes,
    "distinct before": raw.nunique(), "distinct after": signups.nunique(),
})

,dtype before,dtype after,distinct before,distinct after
signup_id,object,object,36,36
student_id,object,object,15,15
full_name,object,object,24,14
email,object,object,18,14
faculty,object,object,13,5
city,object,object,9,3
club,object,object,13,5
signed_up_at,object,datetime64[ns],36,36
fee_paid,object,bool,9,2


## Part B — Duplicates

Part A gave every value one spelling and every column one type. Duplicates come second, so that two copies of a sign-up look the same however they were typed.

### B.1 Exact duplicates

A.1 found 39 rows but only 36 distinct `signup_id`s. **Look.** Are the rows that share an id copies of one submission, or different submissions under the same id? The rows that share an id, side by side:

In [28]:
signups[signups["signup_id"].duplicated(keep=False)].sort_values("signup_id")

,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
34,S015,61-8942,Hana Zaki,hana.zaki@student.guc.edu.eg,Pharmacy,Giza,Football,2026-09-15 13:39:00,False
15,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
24,S022,61-8224,Sara Zaki,sara.zaki@student.guc.edu.eg,EMS,Giza,Music,2026-09-16 01:39:00,True
20,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False
33,S023,58-1125,Ziad Saleh,ziad.saleh@student.guc.edu.eg,IET,Cairo,Chess,2026-09-16 09:29:00,False


In [29]:
repeats_id = signups["signup_id"].duplicated()   # True for the 2nd, 3rd... row with an id
copies_row = signups.duplicated()                  # True for a row identical to an earlier one, in every column
print("rows that repeat a signup_id:        ", repeats_id.sum())
print("rows identical to an earlier row:    ", copies_row.sum())
print("are they the same rows?              ", repeats_id.equals(copies_row))

rows that repeat a signup_id:         3
rows identical to an earlier row:     3
are they the same rows?               True


Three ids repeat (`S015`, `S022`, `S023`), each pair is identical in every column, and these are exactly the rows that `duplicated()` marks. They are the same form submission saved twice, so the copies carry no new information and either one can stay.

**Count and remove.** `duplicated()` marks every copy after the first; `drop_duplicates()` keeps the first and drops the rest.

In [30]:
rows_start = len(raw)
rows_before_b1 = len(signups)
exact_removed = signups.duplicated().sum()

signups = signups.drop_duplicates()
print(f"exact duplicates: {exact_removed} | rows: {rows_before_b1} -> {len(signups)}")

exact duplicates: 3 | rows: 39 -> 36


In [31]:
# ✓ check: no exact copies left, one row per signup_id, and the arithmetic adds up
assert not signups.duplicated().any(), "an exact copy is still there"
assert signups["signup_id"].is_unique, "a signup_id still repeats"
assert len(signups) == rows_before_b1 - exact_removed
print(f"✓ {exact_removed} exact duplicates removed: {rows_before_b1} - {exact_removed} = {len(signups)} rows, one per signup_id")

✓ 3 exact duplicates removed: 39 - 3 = 36 rows, one per signup_id


36 rows, one per `signup_id`. Exact copies are the easy kind, because every column agrees. A student who came back and submitted the same club again got a **new** `signup_id`, so that row is not an exact copy and is still here. B.2 looks for those.

### B.2 The same sign-up submitted again

After B.1 every `signup_id` is unique, but the brief says some students signed up for the same club twice, with different ids. A person is identified by `student_id` (A.3 showed that names and emails are shared), and `club` now has one spelling per club (A.2), so the key is `student_id` + `club`.

**Look.** Every student and club that appears more than once, in time order, with the fee beside it. What changed between the two submissions?

In [32]:
KEY = ["student_id", "club"]
resubmitted = signups[signups.duplicated(KEY, keep=False)].sort_values(KEY + ["signed_up_at"])
print(f"{len(resubmitted)} rows in {resubmitted.groupby(KEY).ngroups} groups of the same student and club")
resubmitted[["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]]

8 rows in 4 groups of the same student and club


,signup_id,student_id,full_name,club,signed_up_at,fee_paid
8,S020,55-9245,Ali Ibrahim,Music,2026-09-15 23:07:00,False
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
22,S021,61-4844,Mohamed Adel,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
35,S017,61-7344,Laila Mostafa,Debate,2026-09-15 18:01:00,False
25,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
21,S032,61-7344,Laila Mostafa,Football,2026-09-17 23:36:00,False
38,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


8 rows in 4 groups, and in all 4 the earlier submission has `fee_paid` False and the later one True: the students came back to mark the fee as paid, as the brief says. The latest submission is the current state of the sign-up. Keeping the first would record 4 paid sign-ups as unpaid, and the club would chase students who have already paid.

**Is the file in time order?** In the Lab 3 order feed the rows arrived in time order, so `keep="last"` kept the newest copy. Here, the row numbers on the left already look out of order: for 61-4844's Debate sign-up, the later `S035` sits at row 3 and the earlier `S021` at row 22.

In [33]:
print("file in time order?", signups["signed_up_at"].is_monotonic_increasing)

kept_by_file_order = set(signups.drop_duplicates(KEY, keep="last")["signup_id"])
kept_by_time = set(signups.sort_values("signed_up_at").drop_duplicates(KEY, keep="last")["signup_id"])
print('keep="last" in file order would keep:', sorted(kept_by_file_order - kept_by_time), "instead of", sorted(kept_by_time - kept_by_file_order))

file in time order? False
keep="last" in file order would keep: ['S017', 'S021'] instead of ['S034', 'S035']


The file is not in time order, and `keep="last"` on it would keep `S017` and `S021`, the earlier and unpaid submissions, in 2 of the 4 groups. "Last" has to mean last in time.

**Fix.** Sort by the parsed `signed_up_at` (A.5 made it a real datetime; as text, `18/09/2026` would sort before `2026-09-17`), then keep the last row of every `student_id` + `club`.

In [34]:
before_b2 = signups.copy()
signups = signups.sort_values("signed_up_at").drop_duplicates(subset=KEY, keep="last")

resubmissions_removed = len(before_b2) - len(signups)
print(f"earlier submissions removed: {resubmissions_removed} | rows: {len(before_b2)} -> {len(signups)}")

earlier submissions removed: 4 | rows: 36 -> 32


### B.3 Check: one row per student per club

In [35]:
# ✓ check: student_id + club is unique, nothing else was lost, and every kept row is the latest of its group
assert not signups.duplicated(KEY).any(), "student_id + club still repeats"
assert len(signups) == before_b2.groupby(KEY).ngroups, "a student-club pair went missing"

latest = before_b2.groupby(KEY)["signed_up_at"].max()
kept = signups.set_index(KEY)["signed_up_at"]
assert (kept == latest.loc[kept.index]).all(), "a kept row is not the latest of its group"
print(f"✓ one row per student per club: {len(signups)} rows, each the latest submission for its student and club")

✓ one row per student per club: 32 rows, each the latest submission for its student and club


In [36]:
# the four groups from the look: the row that stayed
signups[signups["signup_id"].isin(resubmitted["signup_id"])].sort_values(KEY)[["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]]

,signup_id,student_id,full_name,club,signed_up_at,fee_paid
9,S033,55-9245,Ali Ibrahim,Music,2026-09-16 09:07:00,True
3,S035,61-4844,Mohamed Adel,Debate,2026-09-17 11:15:00,True
25,S034,61-7344,Laila Mostafa,Debate,2026-09-16 02:01:00,True
38,S036,61-7344,Laila Mostafa,Football,2026-09-18 23:36:00,True


In all four groups the row that stayed is the later, paid one. Part B in numbers: 39 rows, minus 3 exact duplicates (B.1) = 36, minus 4 earlier submissions of the same student and club (B.2) = 32 rows, one per student per club.

### Save the result

The table passes every check in memory. A file is only right if it reads back right, so we save it, read it back, and check it again.

**Look** at the table as it is now: B.2 sorted it by time, so the rows are out of `signup_id` order and still carry their row labels from `raw`.

In [37]:
print("row labels:", signups.index[:10].tolist(), "...")
print("signup_ids:", signups["signup_id"].head(10).tolist(), "...")

row labels: [17, 14, 10, 4, 7, 11, 6, 2, 23, 1] ...
signup_ids: ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010'] ...


**Save.** Put the rows back in `signup_id` order and drop the old row labels (`index=False`), so the file holds exactly the 9 columns of the brief. pandas would write each line ending in the style of whichever system runs the notebook, so we fix it to `\r\n`, the same as the file we were given: the clean file then comes out byte for byte the same on any machine.

In [38]:
CLEAN_FILE = "club_signups_clean.csv"

signups = signups.sort_values("signup_id").reset_index(drop=True)
signups.to_csv(CLEAN_FILE, index=False, lineterminator="\r\n")
print(f"saved {CLEAN_FILE}: {len(signups)} rows, {signups.shape[1]} columns")

saved club_signups_clean.csv: 32 rows, 9 columns


In [39]:
# ✓ check: the saved file reads back as the same table and keeps every promise
clean = pd.read_csv(CLEAN_FILE)
clean["signed_up_at"] = pd.to_datetime(clean["signed_up_at"], format="%Y-%m-%d %H:%M:%S")

assert list(clean.columns) == list(raw.columns), "the columns changed"
for col, allowed in [("faculty", FACULTIES), ("club", CLUBS), ("city", CITIES)]:
    assert clean[col].isin(allowed).all(), f"{col} reads back with other values"
assert clean["fee_paid"].dtype == bool, f"fee_paid reads back as {clean['fee_paid'].dtype}"
assert not clean.duplicated(KEY).any(), "student_id + club repeats in the file"
pd.testing.assert_frame_equal(clean, signups)   # every value and type, exactly as in memory
print(f"✓ {CLEAN_FILE} reads back identical to the table: {clean.shape[0]} rows, {clean.shape[1]} columns")

✓ club_signups_clean.csv reads back identical to the table: 32 rows, 9 columns


In [40]:
# ✓ check: the raw file is still byte for byte the file we were given (fingerprint recorded before any work)
import hashlib

GIVEN_SHA256 = "f74c6562f7b9061ba3f9b444bae8c3bfefea095276549aa7d49bd3f6d861bce2"
with open("club_signups.csv", "rb") as f:
    assert hashlib.sha256(f.read()).hexdigest() == GIVEN_SHA256, "club_signups.csv has changed"
print("✓ club_signups.csv is unchanged")

✓ club_signups.csv is unchanged


`club_signups_clean.csv` holds 32 rows, one per student per club, with every value spelled one way, and reads back exactly as built. `club_signups.csv` is still the file we were given.

## Part C — The numbers behind the paragraph

The paragraph in `README.md` has to answer two questions with numbers. Both are answered here with experiments on copies of the data: nothing below changes `signups` or the saved file.

### The order of the steps

Would we find the same duplicates if we removed them **before** fixing the club spellings? On a fresh copy of `raw`, nothing fixed yet:

In [41]:
early = raw.drop_duplicates()                # B.1 on the raw data
early_exact = len(raw) - len(early)
early_key = early.duplicated(KEY).sum()       # B.2 with the clubs as typed

print(f"exact duplicates:            {early_exact} before Part A | {exact_removed} after Part A")
print(f"same student + club found:   {early_key} before Part A | {resubmissions_removed} after Part A")
print(f"rows left:                   {len(early) - early_key} before Part A | {len(signups)} after Part A")

exact duplicates:            3 before Part A | 3 after Part A
same student + club found:   2 before Part A | 4 after Part A
rows left:                   34 before Part A | 32 after Part A


Which of the four resubmissions slip through? The club of each submission as it was typed, for the four groups B.2 found:

In [42]:
typed = raw.drop_duplicates().set_index("signup_id")["club"]
order_check = resubmitted.assign(typed=resubmitted["signup_id"].map(typed)).groupby(KEY)["typed"].agg(list).to_frame("club as typed")
order_check["found before the fix?"] = order_check["club as typed"].map(lambda spellings: len(set(spellings)) == 1)
order_check

club as typed  found before the fix?
student_id club                                                  
55-9245    Music            [Music, music]                  False
61-4844    Debate     [Debating, Debating]                   True
61-7344    Debate    [Debate, debate club]                  False
           Football   [football, football]                   True

Exact duplicates come out the same either way (3), because identical rows stay identical whatever we fix. Duplicates found by a **key** depend on that key being clean: with the clubs as typed, only 2 of the 4 resubmissions are found, the two typed the same way twice (`Debating`, `football`). `Music`/`music` and `Debate`/`debate club` slip through, and 34 rows would be left instead of 32. So the spellings must be fixed before the key-based step. The Lab 3 sequence puts duplicates early, and that holds for exact copies, but not for a key that contains a messy column.

### Two students with the same name

A.3 found two students called Mohamed Adel, who also share an email. B.2 identified people by `student_id`. What would a key of name + club, or email + club, have done to the same 36 rows?

In [43]:
person_key = {}
for person in ["full_name", "email"]:
    by_person = before_b2.sort_values("signed_up_at").drop_duplicates([person, "club"], keep="last")
    lost = sorted(set(signups["signup_id"]) - set(by_person["signup_id"]))
    person_key[person] = (len(by_person), lost)
    print(f"{person} + club: {len(by_person)} rows ({len(before_b2) - len(by_person)} removed), wrongly loses {lost}")

full_name + club: 30 rows (6 removed), wrongly loses ['S002', 'S009']
email + club: 30 rows (6 removed), wrongly loses ['S002', 'S009']


In [44]:
# every sign-up of the two Mohamed Adels before B.2
before_b2[before_b2["full_name"] == "Mohamed Adel"].sort_values(["club", "signed_up_at"])[["signup_id", "student_id", "full_name", "faculty", "club", "signed_up_at", "fee_paid"]]

,signup_id,student_id,full_name,faculty,club,signed_up_at,fee_paid
14,S002,55-2992,Mohamed Adel,IET,Debate,2026-09-13 15:38:00,True
22,S021,61-4844,Mohamed Adel,EMS,Debate,2026-09-16 01:15:00,False
3,S035,61-4844,Mohamed Adel,EMS,Debate,2026-09-17 11:15:00,True
2,S008,61-4844,Mohamed Adel,EMS,Music,2026-09-14 23:32:00,True
23,S009,61-4844,Mohamed Adel,EMS,Robotics,2026-09-15 01:05:00,True
18,S024,55-2992,Mohamed Adel,IET,Robotics,2026-09-16 16:16:00,True


Keyed on the name, or on the email, B.2 would remove 6 rows instead of 4 and leave 30. Both Mohamed Adels joined Debate and Robotics, so a name key treats their sign-ups as one person's and keeps only the latest in each club: 55-2992's Debate sign-up (`S002`) and 61-4844's Robotics sign-up (`S009`) would disappear. Keying on `student_id` keeps both students and all their clubs.

### The facts for the paragraph

Every number the README paragraph uses, taken from the variables above:

In [45]:
groups = resubmitted.groupby(KEY).ngroups
paid_later = resubmitted.groupby(KEY)["fee_paid"].agg(lambda fees: (not fees.iloc[0]) and fees.iloc[-1]).sum()
name_rows, name_lost = person_key["full_name"]

facts = pd.Series({
    "faculty / club / city, distinct values": f"{raw['faculty'].nunique()} / {raw['club'].nunique()} / {raw['city'].nunique()} -> {signups['faculty'].nunique()} / {signups['club'].nunique()} / {signups['city'].nunique()}",
    "names / emails for 15 students, distinct": f"{raw['full_name'].nunique()} / {raw['email'].nunique()} -> {signups['full_name'].nunique()} / {signups['email'].nunique()}",
    "fee_paid spellings": raw["fee_paid"].nunique(),
    "signed_up_at formats": f"{is_iso.sum()} rows year-first, {is_slash.sum()} rows day/month/year",
    "rows at the start": rows_start,
    "B.1 exact duplicates removed": exact_removed,
    "B.2 earlier submissions removed": resubmissions_removed,
    "rows left": len(signups),
    "B.2 groups where the later one is paid": f"{paid_later} of {groups}",
    'B.2 groups wrong if keep="last" is unsorted': f"{len(kept_by_file_order - kept_by_time)} of {groups}",
    "before club fix: exact / resubmitted / rows left": f"{early_exact} / {early_key} / {len(early) - early_key}",
    "name + club key: rows left / real sign-ups lost": f"{name_rows} / {', '.join(name_lost)}",
}, name="value")
facts.to_frame()

,value
"faculty / club / city, distinct values",13 / 13 / 9 -> 5 / 5 / 3
"names / emails for 15 students, distinct",24 / 18 -> 14 / 14
fee_paid spellings,9
signed_up_at formats,"25 rows year-first, 14 rows day/month/year"
rows at the start,39
B.1 exact duplicates removed,3
B.2 earlier submissions removed,4
rows left,32
B.2 groups where the later one is paid,4 of 4
"B.2 groups wrong if keep=""last"" is unsorted",2 of 4


In [46]:
# ✓ check: the experiments worked on copies, so the clean table is still what was saved
pd.testing.assert_frame_equal(clean, signups)
print("✓ signups is unchanged by the experiments, and still equal to", CLEAN_FILE)

✓ signups is unchanged by the experiments, and still equal to club_signups_clean.csv
